# Chapter 16: Temperature and Top-k

[Read this chapter online](https://jackluu.io/book/section-5-generation/ch16-temperature-and-topk/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch16-temperature-and-topk.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 16: Temperature and Top-k

![The map with Next-Token Scores and New Text highlighted](https://jackluu.io/book/assets/diagrams/ch16-where-we-are.png)

*Figure 16.1: We tune how the text is generated.*

Random sampling produces diversity, but sometimes at the cost of coherence. The model might select a rare character which violates grammar or produces an unrecognizable word. We’ll want some reins on the randomness. In this chapter you will:

- Use temperature to adjust how risky the model's choices are.
- Use top-k to cut the list of choices short, so the model cannot pick truly bad characters.
- Combine these controls for optimal text generation.

> **Words to Know**
>
> - **Temperature**: a number the scores (logits) are divided by before they become probabilities. A low temperature plays it safe, and a high one takes risks.
> - **Top-k**: a limit that lets the model sample only from the *k* most likely next tokens, where *k* is a number you choose (40 here).

## Theory

There are two knobs between the model's raw scores and the character it ultimately selects. Temperature alters the odds. Top-k pruning determines which characters are eligible. They are independent, and you will usually set both.

![The two generation controls side by side](https://jackluu.io/book/assets/diagrams/ch16-controls.png)

*Figure 16.2: Temperature changes the shape of the odds, and top-k changes how many characters stay in the running.*

### Temperature

Temperature is essentially one mathematical operation applied to logits just before softmax. We divide each logit by one scalar value, and that alters the distribution over characters.

```text
$ python src/examples/ch16_temperature.py
Saved chart: ch16_temperature.png
```

![Bars of probabilities for five example characters at three temperatures](https://jackluu.io/book/assets/plots/ch16_temperature.png)

*Figure 16.3: Low temperature sharpens the choice, and high temperature flattens it.*

- **Low temperature (e.g., 0.5)**: When you divide by a number less than 1, you spread the scores farther apart. The top choice gets heavily favored, and the model plays it safe.
- **High temperature (e.g., 2.0)**: Dividing by a big number pulls the scores together. The probabilities flatten out. Unlikely choices get stronger chances, and the model risks more.
- **T = 1.0**: This is standard sampling, where the model uses its learned probabilities exactly as they are.

### Top-k

There is still a probability at any given temperature that could result in the wrong character being chosen. That chance might be 0.001%. If you sample enough times you will eventually see a mistake.

By using top-k sampling we can cap our selection pool. If we set `top_k = 40`, we would take all 65 character choices, keep the 40 with the highest scores, and drop the bottom 25 by setting their probabilities to zero. By dropping these choices we are preventing any sampling from occurring on those lower end values.

![Ranked scores with the low-scoring tail greyed out](https://jackluu.io/book/assets/diagrams/ch16-top-k.png)

*Figure 16.4: Top-k keeps the highest-scoring characters and zeroes the rest. Drawn here with ten characters and k = 4, while the book's code uses 65 and k = 40.*

These controls adjust the Next-Token Scores on the "where we are" map, right before generation loops back to produce the New Text.

> **In Business**
>
> When creating boilerplate text for compliance documentation with the house-style assistant, you may want to choose a lower temperature (0.5) setting so that the text it generates is always as close to the safe outputs as possible. However, if you want witty taglines to market your business, try using a higher temperature (0.9) setting with top-k sampling (40) to get creative, surprising results that still conform to grammatical norms.

## Code

We apply temperature and top-k right before the softmax function in the generation loop.

```python
# Temperature controls randomness: lower is less random
logits = logits / temperature

# Top-k sampling limits the choices to the k most likely tokens
if top_k is not None:
    threshold = logits.topk(top_k).values[:, -1, None]
    logits = logits.masked_fill(logits < threshold, float("-inf"))

probs   = F.softmax(logits, dim=-1)
next_id = torch.multinomial(probs, num_samples=1)
```

Line 2 divides the scores by the temperature to adjust randomness. Line 7 discards any score below the top-k threshold by setting it to negative infinity.

![Code flow: logits divided by temperature, filtered by top-k, then softmaxed](https://jackluu.io/book/assets/diagrams/ch16-code-flow.png)

*Figure 16.5: The code applies temperature, then top-k, then softmax.*

We implement top-k in PyTorch using `masked_fill`. We take the score of the 40th best token (`threshold`), and change any token with lower score to `-inf`. When we apply softmax to turn scores into probabilities, `-inf` becomes exactly 0.

The run below shows how different settings affect the generated text.

```text
$ python src/ch15_generate_sampling.py
--- temp=0.5, top_k=40 ---
JULIET:
My must grace prove a son the come on my lord.

...
--- temp=0.8, top_k=40 ---
JULIET:
That will, my call thee for thee king win a be parder
The his in suppy ascented a not a say?
...
--- temp=1.0, top_k=40 ---
JULIET:
Look, do indo witned me devise thy grohed:
A give gring brothe whith ighers paison:
...
--- temp=1.5, top_k=40 ---
JULIET:
fattlyals give niet, thou stain, viful-wid thel,
Brob dayse mernion shing you fea,
...
--- temp=1.0, no top_k ---
JULIET:
Than mide smear of thou I am go vious, you me
now you scalf them frate ell, to 'till will she
...
--- Sweet spot ---
temperature=0.8 to 1.0 and top_k=40 usually gives the best results.
```

**What just happened:**

- At `temp=0.5`, the text is coherent but relies heavily on common, safe words.
- At `temp=0.8` to `1.0`, the text feels more natural and varied.
- At `temp=1.5`, the text quickly fills with made-up words ("fattlyals").
- A temperature between 0.8 and 1.0, combined with `top_k=40`, typically works best.

### Shape Check

Table 16.1 lists the shapes of tensors used during top-k filtering.

**Table 16.1:** Tensors modified during the top-k sampling process.

| Tensor | Shape | What it means |
|--------|-------|---------------|
| `threshold` | `[1, 1]` | The cutoff score (the 40th highest value). |
| `logits` | `[1, 65]` | 65 scores, where 25 of them have been set to `-inf`. |

## Try It

> **Try It**
>
> Open the script `src/examples/ch16_explore_temp.py` and run it. It initializes with a very low temperature (`0.1`), and you will see it is essentially doing greedy decoding (picking the safe top character every time).
>
> ```console title="Terminal"
> $ python src/examples/ch16_explore_temp.py
> --- Prompt: 'JULIET:\n' ---
> Generating with T=0.1...
> JULIET:
> I will the shall the shall be the son th...
>
> ```

## Key Takeaways

- Temperature adjusts the spread of the probabilities. Low temperature makes the model conservative, while high temperature makes it creative and risky.
- Top-k restricts the model from ever picking the worst-scoring tokens and so prevents bizarre errors.
- To combine them, first apply temperature, then apply top-k to filter out the bad options, and finally convert to probabilities with softmax.
- A combination of `temperature=0.8` and `top_k=40` is a widely used default for high-quality text generation.

## Check Your Understanding

1. If you set the temperature to 0.1, what happens to the gap between the highest and lowest scores?
2. Why do we replace the eliminated scores in top-k with negative infinity (`-inf`) instead of `0`?
3. Which step must happen first in the code: applying top-k or calculating softmax probabilities?

## Assignment

> **Assignment 16: Turn up the heat**
>
> **About 15 minutes. You change one number and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/examples/ch16_explore_temp.py` as it is. Copy the text the model writes after `JULIET:`.
> 2. **Change one number.** Near the bottom of the file, find `temperature=0.1`. Temperature is the number the scores are divided by before the model draws a character. Change `0.1` to `2.0` and run the script again. The line `Generating with T=0.1...` is a fixed label, so it will still say 0.1.
> 3. **Compare.** Which text keeps repeating the same words? Which one is full of made-up words? Which one is closer to real English?
> 4. **Explain it.** In two or three sentences a manager could follow, say why neither setting would suit an assistant that writes to customers, and which way you would turn the dial from each one.
>
> **Hand in:** both texts, your answers to step 3, and your sentences.

## Further Reading

**Where top-k sampling comes from.** This paper covers story generation based on a prompt. Along the way it introduced the sampling strategy covered in Chapter 16: sample only from the k most likely next tokens. This gives you varied text without allowing the model to randomly choose something nonsensical.

**Why always picking the likeliest word goes wrong.** Simply selecting the next token with the highest score at each time step, known as greedy decoding in Chapter 15, yields bland and repetitive output. The paper shows that this is the case because natural language texts are not composed of the locally most predictable word at each time step. They propose an alternative that samples from the smallest set of tokens with probabilities that sum to a given proportion. Choice of next token is just as important as training.


Fan, A., Lewis, M., & Dauphin, Y. (2018). *Hierarchical neural story generation* (arXiv:1805.04833). arXiv. https://doi.org/10.48550/arXiv.1805.04833

Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2019). *The curious case of neural text degeneration* (arXiv:1904.09751). arXiv. https://doi.org/10.48550/arXiv.1904.09751

---

### `src/ch15_generate_sampling.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch15_generate_sampling.py"   # a cell has none, and the file uses it to find the text

"""
Explore temperature and top-k sampling.
This file belongs to Chapter 16.
Run: python src/ch15_generate_sampling.py
"""
import os
import sys
import torch
import torch.nn.functional as F


from src.utils.config import GPTConfig
from src.ch09_gpt_model import GPT

# Settings
CHECKPOINT_PATH = "checkpoints/model.pt"
DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

# --- The Idea ---

def generate(
    model, prompt, encode, decode, cfg, max_new_tokens=200,
    temperature=1.0, top_k=None
):
    ids = torch.tensor([encode(prompt)], dtype=torch.long)

    for _ in range(max_new_tokens):
        ctx    = ids[:, -cfg.block_size:]
        logits = model(ctx)
        logits = logits[:, -1, :]

        # Temperature controls randomness: lower is less random
        logits = logits / temperature

        # Top-k sampling limits the choices to the k most likely tokens
        if top_k is not None:
            threshold = logits.topk(top_k).values[:, -1, None]
            logits = logits.masked_fill(logits < threshold, float("-inf"))

        probs   = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids     = torch.cat([ids, next_id], dim=1)

    return decode(ids[0].tolist())

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 16: Temperature and Top-k Sampling\n")

    if not os.path.exists(DATA_PATH):
        print("ERROR: Run: python src/utils/download_data.py")
        sys.exit(1)

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()

    chars  = sorted(set(text))
    char_to_id   = {ch: i for i, ch in enumerate(chars)}
    id_to_char   = {i: ch for i, ch in enumerate(chars)}

    encode = lambda s: [char_to_id[c] for c in s if c in char_to_id]
    decode = lambda ids: "".join([id_to_char[i] for i in ids])

    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=False
    )
    cfg   = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    print(f"Model loaded (trained {checkpoint['step']} steps)\n")

    prompt = "JULIET:\n"
    n_tokens = 150

    print(f"Prompt: {repr(prompt)}")
    print(f"Generating {n_tokens} characters each...\n")

    configs = [
        {"temperature": 0.5, "top_k": 40,   "label": "temp=0.5, top_k=40"},
        {"temperature": 0.8, "top_k": 40,   "label": "temp=0.8, top_k=40"},
        {"temperature": 1.0, "top_k": 40,   "label": "temp=1.0, top_k=40"},
        {"temperature": 1.5, "top_k": 40,   "label": "temp=1.5, top_k=40"},
        {"temperature": 1.0, "top_k": None, "label": "temp=1.0, no top_k"},
    ]

    with torch.no_grad():
        for cfg_dict in configs:
            text_out = generate(
                model, prompt, encode, decode, cfg,
                max_new_tokens=n_tokens,
                temperature=cfg_dict["temperature"],
                top_k=cfg_dict["top_k"]
            )
            print("---", cfg_dict["label"], "---")
            print(text_out)
            print()

    print("--- Sweet spot ---")
    print("temperature=0.8 to 1.0 and top_k=40 usually gives the best results.")

---

### `src/examples/ch16_explore_temp.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch16_explore_temp.py"   # a cell has none, and the file uses it to find the text

"""Generate text with a very low temperature to simulate greedy decoding."""
import os
import sys
import torch

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

from src.ch03_tokenizer import build_vocab, encode, decode
from src.ch09_gpt_model import GPT
from src.ch15_generate_sampling import generate

def main():
    torch.manual_seed(42)
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
    )
    
    if not os.path.exists(checkpoint_path):
        print("Checkpoint not found.")
        return
        
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    
    cfg = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    
    text_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "src", "data", "shakespeare.txt"
    )
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()
    chars, char_to_id, id_to_char = build_vocab(text)
    
    encode_fn = lambda s: encode(s, char_to_id)
    decode_fn = lambda ids: decode(ids, id_to_char)
    
    prompt = "JULIET:\n"
    print(f"--- Prompt: {repr(prompt)} ---")
    print("Generating with T=0.1...")
    
    # Using the generation function that takes temperature and top_k
    output = generate(
        model, prompt, encode_fn, decode_fn, cfg, 
        max_new_tokens=40, temperature=0.1, top_k=40
    )
    print(output + "...\n")

if __name__ == "__main__":
    main()

---

### `src/examples/ch16_temperature.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch16_temperature.py"   # a cell has none, and the file uses it to find the text

"""Plot the effect of temperature on a probability distribution."""
import matplotlib.pyplot as plt
import numpy as np

def main():
    # Example scores (logits)
    logits = np.array([2.5, 1.8, 0.5, -1.0, -2.5])
    labels = ["A", "B", "C", "D", "E"]
    
    temps = [0.5, 1.0, 2.0]
    
    fig, axes = plt.subplots(1, 3, figsize=(6.5, 3.5), dpi=300)
    fig.patch.set_facecolor('white')
    
    for ax, T in zip(axes, temps):
        # Apply temperature
        scaled = logits / T
        # Softmax
        exp_L = np.exp(scaled - np.max(scaled))
        probs = exp_L / np.sum(exp_L)
        
        ax.bar(labels, probs, color="#00695C")
        ax.set_ylim(0, 1.0)
        ax.set_title(f"T = {T}")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig("ch16_temperature.png")
    print("Saved chart: ch16_temperature.png")

if __name__ == "__main__":
    main()